# Example-count-matched main ablations (one fixed seed)

This notebook removes the training-example-count confound from the aspect
decomposition and candidate-pool augmentation ablations. Run one mode per clean
kernel by setting `ABLATION_MODE` to `full`, `no_pool_augmentation`, or
`no_aspect_decomposition`.

| Mode | Source rows | Distinct pools per row | Before matching | Final examples |
|---|---:|---:|---:|---:|
| `full` | 1,498 aspect queries | 3 | 4,494 | 4,494 |
| `no_pool_augmentation` | 1,498 aspect queries | 1 | 1,498 | 4,494 (3 repetitions each) |
| `no_aspect_decomposition` | 100 full queries | 3 | 300 | 4,494 (14–15 repetitions each) |

All modes use seed 42, two epochs, batch size 2, and the same optimizer and
scheduler. On one GPU this gives 4,494 optimizer steps for every condition.
The notebook also records total non-padding tokens, since example-count matching
does not imply exact token-count matching.

The pool-augmentation contrast compares three distinct pools against three
exposures to one pool. The aspect contrast compares aspect-specific supervision
against repeated full-query supervision. The latter treats aspect decomposition
as the complete query-and-label construction operation; it does not isolate query
wording from max-aggregated full-query labels.


In [ ]:
pip install transformers datasets peft trl bitsandbytes accelerate ipywidgets tensorboard optuna

In [ ]:
pip install unsloth

In [ ]:
import os
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"

In [ ]:
import os, re
import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import random
import numpy as np
import torch
from transformers import set_seed

In [ ]:
MAX_TOKENS=16_384

In [ ]:
RUN_SEED = 42
ABLATION_MODES = (
    "full",
    "no_pool_augmentation",
    "no_aspect_decomposition",
)
ABLATION_MODE = os.environ.get("ABLATION_MODE", "full")
TARGET_TRAIN_EXAMPLES = 4_494

if ABLATION_MODE not in ABLATION_MODES:
    raise ValueError(
        f"ABLATION_MODE must be one of {ABLATION_MODES}, got {ABLATION_MODE!r}."
    )

print(
    f"Matched ablation run: mode={ABLATION_MODE}, "
    f"examples={TARGET_TRAIN_EXAMPLES}, seed={RUN_SEED}"
)


In [ ]:
random.seed(RUN_SEED)
np.random.seed(RUN_SEED)
torch.manual_seed(RUN_SEED)
torch.cuda.manual_seed_all(RUN_SEED)
set_seed(RUN_SEED)

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "meta-llama/Llama-3.1-8B-Instruct",
    max_seq_length = MAX_TOKENS,
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    load_in_16bit = True,
    full_finetuning = False, # [NEW!] We have full finetuning now!
)

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 128,           # Larger = higher accuracy, but might overfit
    lora_alpha = 256,  # Recommended alpha == r at least
    lora_dropout = 0.05,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias = "none",
    random_state = RUN_SEED,
)

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

In [ ]:
from datasets import Dataset, load_from_disk
from pathlib import Path
import pandas as pd


def find_doris_data_dir():
    candidates = (
        Path("splits"),
        Path("DorisMAE"),
        Path("DorisMAE/splits"),
    )
    required = ("Query.json", "Annotation.json", "Corpus.json")
    for candidate in candidates:
        if all((candidate / filename).exists() for filename in required):
            return candidate
    raise FileNotFoundError(
        "Could not find Query.json, Annotation.json, and Corpus.json together "
        "under splits/, DorisMAE/, or DorisMAE/splits/."
    )


def find_aspect_dataset_path():
    candidates = (
        Path("reranking_dataset_doris_mae"),
        Path("DorisMAE/reranking_dataset_doris_mae"),
    )
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the saved reranking_dataset_doris_mae dataset."
    )


def construct_no_aspect_dataset(query_path, annotation_path):
    """Create one full-query row per parent using max relevance over its aspects."""
    query_df = pd.read_json(query_path)
    annotations_df = pd.read_json(annotation_path)
    annotations_df["abstract_id"] = annotations_df["abstract_id"].astype(int)
    annotations_df["aspect_id"] = annotations_df["aspect_id"].astype(str)

    # This reproduces the original pandas last-write behavior for duplicate keys.
    annotation_lookup = annotations_df.set_index(
        ["abstract_id", "aspect_id"]
    )["score"].to_dict()

    rows = []
    missing_keys = []
    for query_index, query in query_df.iterrows():
        candidate_pool = [int(doc_id) for doc_id in query["candidate_pool"]]
        aspect_ids = set()
        for main_aspect_id, sub_aspect_ids in query["aspects"].items():
            aspect_ids.add(str(main_aspect_id))
            aspect_ids.update(str(aspect_id) for aspect_id in sub_aspect_ids)

        candidate_scores = []
        for abstract_id in candidate_pool:
            keys = [(abstract_id, aspect_id) for aspect_id in aspect_ids]
            absent = [key for key in keys if key not in annotation_lookup]
            if absent:
                missing_keys.extend(absent[: max(0, 20 - len(missing_keys))])
                candidate_scores.append(-1)
            else:
                candidate_scores.append(
                    max(int(annotation_lookup[key]) for key in keys)
                )

        rows.append({
            "query_text": query["query_text"],
            "aspect_id": "original_query",
            "original_query_id": int(query_index),
            "original_query_type": query["query_type"],
            "candidate_pool": candidate_pool,
            "candidate_scores": candidate_scores,
        })

    if missing_keys:
        raise ValueError(
            "Missing annotations encountered while aggregating full-query labels. "
            f"First examples: {missing_keys[:20]}"
        )

    return Dataset.from_pandas(pd.DataFrame(rows), preserve_index=False)


DORIS_DATA_DIR = find_doris_data_dir()
if ABLATION_MODE == "no_aspect_decomposition":
    dataset = construct_no_aspect_dataset(
        DORIS_DATA_DIR / "Query.json",
        DORIS_DATA_DIR / "Annotation.json",
    )
else:
    dataset = load_from_disk(find_aspect_dataset_path())

print(f"Loaded source rows for {ABLATION_MODE}: {len(dataset)}")


In [ ]:
# Apply the same zero-positive eligibility rule to every condition.
dataset = dataset.filter(
    lambda example: any(int(score) > 0 for score in example["candidate_scores"]),
    desc="Removing zero-positive queries",
)
dataset = dataset.map(
    lambda _example, index: {"source_row_id": int(index)},
    with_indices=True,
    desc="Recording stable source row IDs",
)

EXPECTED_SOURCE_ROWS = {
    "full": 1_498,
    "no_pool_augmentation": 1_498,
    "no_aspect_decomposition": 100,
}
assert len(dataset) == EXPECTED_SOURCE_ROWS[ABLATION_MODE], (
    f"Expected {EXPECTED_SOURCE_ROWS[ABLATION_MODE]} source rows for "
    f"{ABLATION_MODE}, found {len(dataset)}."
)
print(f"Eligible source rows: {len(dataset)}")


In [ ]:
SOURCE_PREVIEW_INDEX = min(100, len(dataset) - 1)
dataset[SOURCE_PREVIEW_INDEX]


In [ ]:
import pandas as pd

corpus_df = pd.read_json(DORIS_DATA_DIR / "Corpus.json")


In [ ]:
corpus_df.head()

In [ ]:
len(corpus_df)

In [ ]:
print("Creating abstract_id -> text lookup dictionary...")
abstract_lookup = {}
for _, row in corpus_df.iterrows():
    title = row['title'].replace("_", " ")
    text = f"Title: {title}\nAbstract: {row['original_abstract']}"
    abstract_lookup[int(row['abstract_id'])] = text

print(f"Created lookup for {len(abstract_lookup)} abstracts.")

In [ ]:
abstract_lookup[0]

In [ ]:
import random, hashlib
from typing import List, Tuple, Dict, Any

# -----------------------------
# CONFIG
# -----------------------------
N_VARIANTS = 1 if ABLATION_MODE == "no_pool_augmentation" else 3
# full/no-aspect: three distinct pools; no-pool: one pool later repeated.
TARGET_SIZE      = 50
MAX_POSITIVES    = 5
HARD_NEG_TOP     = 100    # set None to use ALL negatives
MAX_TOTAL_TRIES  = 3000  # per query; stop early if can't find more unique pools

# Uniqueness definition:
# - set-based (order-invariant): same 20 doc_ids in different order counts as DUPLICATE
def pool_signature(doc_ids: List[Any]) -> Tuple[int, ...]:
    return tuple(sorted(int(d) for d in doc_ids))

# If you ever want "unique-by-order" instead, use:
# def pool_signature(doc_ids): return tuple(int(d) for d in doc_ids)


def stable_seed(row_id: int, query_text: str) -> int:
    # row_id prevents collisions if query_text repeats
    h = hashlib.md5(f"{row_id}||{query_text}".encode("utf-8")).hexdigest()
    return int(h[:8], 16)


# -----------------------------
# PROMPT + TARGET CREATION
# -----------------------------
def create_single_conversation_entry(
    query_text: str,
    selected_docs: List[Tuple[Any, int]],
    abstract_lookup: Dict[int, str],
) -> Dict[str, Any]:
    """
    selected_docs: list of (doc_id, score), already shuffled in input order
    Returns dict with:
      - conversations
      - pool_k, scores_k
    """
    passages_str_list = []
    scores_k = []
    pool_k = []

    for i, (doc_id, score) in enumerate(selected_docs):
        pool_k.append(doc_id)
        scores_k.append(int(score))

        doc_text = abstract_lookup.get(int(doc_id), "Error: Document text not found.")
        doc_text = " ".join(doc_text.split()[:300])  # truncate to ~300 words
        # doc_text = " ".join(doc_text.split())  # truncate to ~300 words
        passages_str_list.append(f"[{i+1}] {doc_text}")

    passages_for_prompt = "\n\n---\n\n".join(passages_str_list)

    user_msg = (
        f"You are an expert academic paper reranker. "
        f"Your task is to re-order the given list of passages (from [1] to [{len(selected_docs)}]) "
        f"based on their relevance to the query. Respond with only the ranking and nothing else.\n\n"
        f"Example output for 8 passages:\n"
        f"[2] > [5] > [4] > [8] > [6] > [1] > [3] > [7]\n\n"
        f"Query: {query_text}\n\n"
        f"Passages:\n{passages_for_prompt}\n\n"
        f"Your ranking (most to least relevant):"
    )

    # repeated_user_message = user_msg + "\n\n" + user_msg

    # Ground truth ranking: score desc, tie broken by earlier index (stable)
    indexed_scores = [(i+1, s) for i, s in enumerate(scores_k)]
    sorted_indices = sorted(indexed_scores, key=lambda x: (x[1], -x[0]), reverse=True)
    target_str = " > ".join(f"[{idx}]" for idx, _ in sorted_indices)

    conversation = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": target_str},
    ]

    return {"conversations": conversation, "pool_k": pool_k, "scores_k": scores_k}


# -----------------------------
# UNIQUE POOL SAMPLING
# -----------------------------
def sample_unique_pools_for_query(
    example: Dict[str, Any],
    *,
    rng: random.Random,
    n_variants: int,
    target_size: int,
    max_positives: int,
    hard_neg_top: int | None,
    max_total_tries: int,
) -> List[List[Tuple[Any, int]]]:
    pool = example["candidate_pool"]
    scores = example["candidate_scores"]

    # Bucket
    neg  = [(d, int(s)) for d, s in zip(pool, scores) if int(s) == 0]
    pos1 = [(d, int(s)) for d, s in zip(pool, scores) if int(s) == 1]
    pos2 = [(d, int(s)) for d, s in zip(pool, scores) if int(s) >= 2]
    all_pos = pos2 + pos1
    all_docs = all_pos + neg

    if hard_neg_top is not None and len(neg) > hard_neg_top:
        neg = neg[:hard_neg_top]

    # If pool is tiny, we'll just return <=1 variant from what exists.
    if len(all_docs) == 0:
        return []

    # Try structured modes first, then keep sampling "any" for coverage
    modes = ["only_2", "only_1", "mixed", "any", "any", "any"]

    seen = set()
    unique_selected_docs: List[List[Tuple[Any, int]]] = []

    tries = 0
    while len(unique_selected_docs) < n_variants and tries < max_total_tries:
        mode = modes[tries % len(modes)]
        tries += 1

        # ---- choose positives
        positives: List[Tuple[Any, int]] = []
        if mode == "only_2":
            k = min(len(pos2), max_positives)
            positives = rng.sample(pos2, k) if k > 0 else []
        elif mode == "only_1":
            k = min(len(pos1), max_positives)
            positives = rng.sample(pos1, k) if k > 0 else []
        elif mode == "mixed":
            k2 = min(len(pos2), max_positives // 2)
            k1 = min(len(pos1), max_positives - k2)
            if k2 > 0: positives += rng.sample(pos2, k2)
            if k1 > 0: positives += rng.sample(pos1, k1)
        else:  # "any"
            k = min(len(all_pos), max_positives)
            positives = rng.sample(all_pos, k) if k > 0 else []

        # ---- choose negatives
        needed = target_size - len(positives)
        if needed < 0:
            positives = positives[:target_size]
            needed = 0

        negatives: List[Tuple[Any, int]] = []
        if needed > 0:
            kneg = min(len(neg), needed)
            negatives = rng.sample(neg, kneg) if kneg > 0 else []

        if not positives and all_pos and max_positives > 0:
            positives = [rng.choice(all_pos)]

        selected = positives + negatives

        # ---- fill up to target_size with remaining docs (pos or neg) not already included
        if len(selected) < target_size:
            used_ids = {d for d, _ in selected}
            remaining = [p for p in all_docs if p[0] not in used_ids]
            fill = remaining[: max(0, target_size - len(selected))]
            selected += fill

        # If still < target_size because the overall pool is too small, that's fine:
        # we'll produce a smaller list (but signature uniqueness still applies).
        doc_ids = [d for d, _ in selected]
        sig = pool_signature(doc_ids)

        if sig in seen:
            continue

        seen.add(sig)
        rng.shuffle(selected)  # IMPORTANT: avoid positional shortcut
        unique_selected_docs.append(selected)

    return unique_selected_docs


# -----------------------------
# DATASET MAP FUNCTION (EXPANDS ROWS)
# -----------------------------
def formatting_prompts_func_unique(examples, indices):
    """
    Requires dataset.map(..., batched=True, with_indices=True)
    Assumes these exist in scope:
      - tokenizer
      - abstract_lookup
    """
    out_text, out_pools, out_scores, out_source_keys = [], [], [], []
    n = len(examples["query_text"])

    for i in range(n):
        ex = {k: examples[k][i] for k in examples}
        row_id = int(ex["source_row_id"])
        qtext = ex["query_text"]

        rng = random.Random(stable_seed(row_id, qtext))

        unique_pools = sample_unique_pools_for_query(
            ex,
            rng=rng,
            n_variants=N_VARIANTS,
            target_size=TARGET_SIZE,
            max_positives=MAX_POSITIVES,
            hard_neg_top=HARD_NEG_TOP,
            max_total_tries=MAX_TOTAL_TRIES,
        )

        for variant_index, selected_docs in enumerate(unique_pools):
            processed = create_single_conversation_entry(
                query_text=qtext,
                selected_docs=selected_docs,
                abstract_lookup=abstract_lookup,
            )

            formatted = tokenizer.apply_chat_template(
                processed["conversations"],
                tokenize=False,
                add_generation_prompt=False,
            ).removeprefix("<bos>")

            out_text.append(formatted)
            out_pools.append(processed["pool_k"])
            out_scores.append(processed["scores_k"])
            out_source_keys.append(f"{row_id}:{variant_index}")

    return {
        "text": out_text,
        "candidate_pool_k": out_pools,
        "candidate_scores_k": out_scores,
        "source_example_key": out_source_keys,
    }


# def has_positives(example):
#     return any(int(s) > 0 for s in example["candidate_scores"])

# dataset_filtered = dataset.filter(has_positives)

# print(f"Original: {len(dataset)}, After filtering: {len(dataset_filtered)}")
# print(f"Removed {len(dataset) - len(dataset_filtered)} queries with zero positives")

augmented_dataset = dataset.map(
    formatting_prompts_func_unique,
    batched=True,
    with_indices=True,
    remove_columns=dataset.column_names,
    load_from_cache_file=False,
)

print("Original size:", len(dataset))
print("Augmented size:", len(augmented_dataset))


# -----------------------------
# EXACT EXAMPLE-COUNT MATCHING
# -----------------------------
from collections import Counter

import random

def balanced_repeat_indices(
    *,
    source_count: int,
    target_count: int,
    seed: int,
) -> list[int]:
    """Repeat all source examples as evenly as possible to an exact target size."""
    if source_count <= 0:
        raise ValueError("source_count must be positive.")
    if target_count < source_count:
        raise ValueError(
            f"target_count {target_count} is smaller than source_count {source_count}; "
            "matching must not discard source examples."
        )

    repetitions, remainder = divmod(target_count, source_count)
    indices = [
        source_index
        for source_index in range(source_count)
        for _ in range(repetitions)
    ]

    rng = random.Random(seed)
    remainder_order = list(range(source_count))
    rng.shuffle(remainder_order)
    indices.extend(remainder_order[:remainder])
    rng.shuffle(indices)
    return indices


base_training_dataset = augmented_dataset
EXPECTED_BASE_EXAMPLES = {
    "full": 4_494,
    "no_pool_augmentation": 1_498,
    "no_aspect_decomposition": 300,
}
assert len(base_training_dataset) == EXPECTED_BASE_EXAMPLES[ABLATION_MODE], (
    f"Expected {EXPECTED_BASE_EXAMPLES[ABLATION_MODE]} generated examples for "
    f"{ABLATION_MODE}, found {len(base_training_dataset)}."
)

matching_indices = balanced_repeat_indices(
    source_count=len(base_training_dataset),
    target_count=TARGET_TRAIN_EXAMPLES,
    seed=RUN_SEED,
)
matched_training_dataset = base_training_dataset.select(matching_indices)
repetition_counts = Counter(matching_indices)
repetition_histogram = dict(sorted(Counter(repetition_counts.values()).items()))

assert len(matched_training_dataset) == TARGET_TRAIN_EXAMPLES
assert max(repetition_counts.values()) - min(repetition_counts.values()) <= 1

print("Source rows:", len(dataset))
print("Generated examples before matching:", len(base_training_dataset))
print("Final matched examples:", len(matched_training_dataset))
print("Repetition histogram (repetitions -> base examples):", repetition_histogram)


In [ ]:
SAMPLE_INDEX = min(100, len(matched_training_dataset) - 1)
matched_training_dataset[SAMPLE_INDEX]


In [ ]:
print(matched_training_dataset[SAMPLE_INDEX]["text"])


In [ ]:
bad_items = [
    (index, item)
    for index, item in enumerate(matched_training_dataset)
    if not any(int(score) > 0 for score in item["candidate_scores_k"])
]
assert not bad_items, f"Matched data contains zero-positive examples: {bad_items[:5]}"
print("All matched training examples contain at least one positive document.")


In [ ]:
import json

ablation_manifest = {
    "experiment": "example_count_matched_main_ablations",
    "ablation_mode": ABLATION_MODE,
    "run_seed": RUN_SEED,
    "matching_basis": "formatted training-example count",
    "target_training_examples": TARGET_TRAIN_EXAMPLES,
    "source_query_rows": len(dataset),
    "pool_variants_per_source_row": N_VARIANTS,
    "base_examples_before_matching": len(base_training_dataset),
    "unique_base_examples": len(repetition_counts),
    "minimum_repetitions_per_base_example": min(repetition_counts.values()),
    "maximum_repetitions_per_base_example": max(repetition_counts.values()),
    "repetition_histogram": {
        str(repetitions): count
        for repetitions, count in repetition_histogram.items()
    },
    "target_candidate_count": TARGET_SIZE,
    "max_positives": MAX_POSITIVES,
    "epochs": 2,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 1,
    "expected_single_gpu_optimizer_steps": 4_494,
}
print(json.dumps(ablation_manifest, indent=2))


In [ ]:
matched_training_dataset[SAMPLE_INDEX]


In [ ]:
print(
    f"Example of matched {ABLATION_MODE} training data:\n"
    f"{matched_training_dataset[SAMPLE_INDEX]['text']}"
)


In [ ]:
def calculate_length(example):
    tokens = tokenizer(text=example["text"], truncation=False)
    return {"token_length": len(tokens["input_ids"])}


print("Calculating token lengths for the matched training set...")
dataset_with_lengths = matched_training_dataset.map(calculate_length, num_proc=4)
all_lengths = dataset_with_lengths["token_length"]
ablation_manifest["token_accounting"] = {
    "total_nonpadding_tokens_per_epoch": int(sum(all_lengths)),
    "mean_nonpadding_tokens_per_example": float(sum(all_lengths) / len(all_lengths)),
    "maximum_tokens_in_example": int(max(all_lengths)),
}
print(json.dumps(ablation_manifest["token_accounting"], indent=2))


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = matched_training_dataset,
    eval_dataset = None,
    args = SFTConfig(
        max_seq_length = MAX_TOKENS,
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 1, # Use GA to mimic batch size!
        warmup_steps = 20,
        # gradient_checkpointing = True,
        num_train_epochs = 2, # Set this for 1 full training run.
        # max_steps = 30,
        learning_rate = 1e-5, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_torch",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = RUN_SEED,
        report_to = "none", # Use TrackIO/WandB etc
        bf16=True
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part    = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

In [ ]:
tokenizer.decode(trainer.train_dataset[SAMPLE_INDEX]["input_ids"])


In [ ]:
tokenizer.decode([
    tokenizer.pad_token_id if token_id == -100 else token_id
    for token_id in trainer.train_dataset[SAMPLE_INDEX]["labels"]
]).replace(tokenizer.pad_token, " ")


In [ ]:
trainer_stats = trainer.train()

In [ ]:
from datetime import datetime
import json
from pathlib import Path

base_model_name = "Llama-3.1-8B-Instruct"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_directory = Path(
    f"models/{base_model_name}_matched_ablation_{ABLATION_MODE}"
    f"_examples_{TARGET_TRAIN_EXAMPLES}_seed_{RUN_SEED}/{timestamp}"
)

model.save_pretrained_merged(
    str(run_directory),
    tokenizer,
    save_method="merged_16bit",
)

ablation_manifest["checkpoint_directory"] = str(run_directory)
ablation_manifest["actual_visible_cuda_devices"] = torch.cuda.device_count()
manifest_path = run_directory / "matched_ablation_manifest.json"
manifest_path.write_text(
    json.dumps(ablation_manifest, indent=2) + "\n",
    encoding="utf-8",
)
print(f"Saved matched-ablation manifest to {manifest_path}")


In [ ]:
# import json

# bad = []
# for k, v in model.config.to_dict().items():
#     try:
#         json.dumps(v)
#     except TypeError:
#         bad.append((k, type(v), v))
# bad[:20], len(bad)


In [ ]:
# qc = model.config.quantization_config

# # defensive: qc might be a dict or a BitsAndBytesConfig-like object
# if isinstance(qc, dict):
#     qc.pop("get_loading_attributes", None)
# else:
#     # if it's an object, remove the attribute if present
#     if hasattr(qc, "get_loading_attributes"):
#         delattr(qc, "get_loading_attributes")

# # also: safest to ensure it's plain-JSON-able
# model.config.quantization_config = qc

# from datetime import datetime
# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# model.save_pretrained_merged(f"models/{timestamp}/gemma-3-finetune", tokenizer)


In [ ]:
# model.save_pretrained(f"models/{timestamp}/llama_varaints_3_target_size_50_positives_1")  # Local saving
# tokenizer.save_pretrained(f"models/{timestamp}/llama_varaints_3_target_size_50_positives_1")